# Entrenament de maia a Google Cloud (L4)

Aquest notebook s'executa **al teu ordinador**. No entrena aquí: prepara el dataset, el puja a Cloud Storage i
engega una VM amb una GPU **NVIDIA L4** que entrena amb **QLoRA (Unsloth)** i s'apaga sola en acabar.

Flux:

1. **Configuració** — model, variant del dataset (K, KP, KPS) i hiperparàmetres.
2. **Preparació** (només el primer cop) — APIs, bucket i compte de servei.
3. **Exporta** el dataset amb `scripts/exporta_dataset.py`.
4. **Puja** el run i **engega** la VM.
5. **Segueix** el log fins que acabi.
6. **Descarrega** l'adaptador, les mètriques i les mostres a `runs/<run>/`.
7. **Esborra** la VM.

Requisits locals: `gcloud` autenticat amb un compte que tingui permisos al projecte `maia-andorra`.

Cost: la VM és **spot** per defecte (molt més barata, però Google la pot aturar). Si s'atura, la cel·la
«Reprèn» la torna a engegar i l'entrenament continua des de l'últim checkpoint. Hi ha un límit dur
d'hores (`MAX_HOURS`): passat aquest temps la VM s'atura igualment.

## 1. Configuració

In [ ]:
import datetime, json, os, shutil, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
assert (ROOT / "scripts" / "exporta_dataset.py").exists(), f"Obre el notebook des de 03-PROJECTS/maia (ara: {ROOT})"

PROJECT = "maia-andorra"
REGION = "europe-west4"                      # regió del bucket
# Zones amb L4 on hi ha quota (1 GPU a cada regió; el límit global també és 1). Es proven per ordre:
# les L4 sovint s'esgoten en una regió sencera (passat el 2026-10-27 a tot europe-west4).
ZONES = [
    "europe-west4-a", "europe-west4-b", "europe-west4-c",
    "europe-west1-b", "europe-west1-c", "europe-west3-a", "europe-west3-b",
    "europe-west2-a", "europe-west2-b", "europe-west6-b", "europe-west6-c",
    "europe-west9-a", "europe-west9-b", "europe-west9-c",
    "europe-southwest1-a", "europe-southwest1-b", "europe-southwest1-c",
]
BUCKET = "maia-andorra-entrenaments"
SERVICE_ACCOUNT = f"maia-entrenament@{PROJECT}.iam.gserviceaccount.com"

MODELS = {
    "e4b": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    "31b": "unsloth/gemma-4-31B-it-unsloth-bnb-4bit",   # QLoRA ~22 GB segons Unsloth: just a la L4
}
MODEL = "e4b"
VARIANT = "K"            # K = coneixement · KP = + Parla real · KPS = + Parla sintètica
# True = 30 passos per comprovar que tot funciona; False = entrenament complet.
# Sense obrir el notebook: MAIA_PROVA=0 jupyter nbconvert --to notebook --execute training/entrena.ipynb
PROVA = os.environ.get("MAIA_PROVA", "1") != "0"
SPOT = True
MAX_HOURS = 10

CONFIG = {
    "model": MODELS[MODEL],
    "variant": VARIANT,
    "chat_template": "gemma-4" if MODEL == "e4b" else "gemma-4-thinking",
    "max_seq_length": 2048,
    "lora_r": 32,
    "lora_alpha": 32,
    "epochs": 2,
    "learning_rate": 2e-4,
    "batch_size": 4,
    "grad_accum": 4,          # batch efectiu = 16
    "eval_steps": 200,
    "seed": 3407,
    "export_gguf": False,
    # Unsloth amb Gemma 4: la compilació per blocs trenca el gradient checkpointing
    # (CheckpointError al primer pas, unsloth#14063). Vegeu la fitxa FTD.
    "env": {"UNSLOTH_BLOCK_COMPILE": "0"},
    "prompts": [
        "Què és el Consell de la Terra?",
        "Quines són les set parròquies d'Andorra?",
        "Explica'm què era la Querimònia.",
        "Què es menja per Sant Jordi a Andorra?",
        "Com va anar la reforma constitucional del 1993?",
    ],
}
if PROVA:
    CONFIG["max_steps"] = 30
    CONFIG["eval_steps"] = 15

RUN = f"{datetime.datetime.now():%Y%m%d-%H%M}-{MODEL}-{VARIANT.lower()}" + ("-prova" if PROVA else "")
CONFIG["run"] = RUN
VM = f"maia-{RUN}"
GS = f"gs://{BUCKET}/runs/{RUN}"
print("Run:", RUN)
print("VM: ", VM)

In [ ]:
def sh(cmd, check=True, quiet=False):
    """Executa una ordre i en retorna la sortida. Sempre contra PROJECT, mai el projecte per defecte de gcloud."""
    if not quiet:
        print("$", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    if check and res.returncode != 0:
        raise RuntimeError(res.stderr.strip() or res.stdout.strip())
    return res.stdout.strip()

def gcloud(*args, **kw):
    return sh(["gcloud", *args, f"--project={PROJECT}"], **kw)

## 2. Preparació (només el primer cop)

Activa les APIs, crea el bucket i un compte de servei propi per a la VM, amb permís només sobre aquest bucket.
Es pot tornar a executar: el que ja existeix no es toca.

In [ ]:
gcloud("services", "enable", "compute.googleapis.com", "storage.googleapis.com", "iam.googleapis.com")

if gcloud("storage", "buckets", "list", f"--filter=name={BUCKET}", "--format=value(name)", quiet=True) != BUCKET:
    gcloud("storage", "buckets", "create", f"gs://{BUCKET}", f"--location={REGION}", "--uniform-bucket-level-access")

sa_name = SERVICE_ACCOUNT.split("@")[0]
if not gcloud("iam", "service-accounts", "list", f"--filter=email={SERVICE_ACCOUNT}", "--format=value(email)", quiet=True):
    gcloud("iam", "service-accounts", "create", sa_name, "--display-name=maia entrenament")
    time.sleep(10)  # el compte tarda uns segons a existir per a IAM

gcloud("storage", "buckets", "add-iam-policy-binding", f"gs://{BUCKET}",
       f"--member=serviceAccount:{SERVICE_ACCOUNT}", "--role=roles/storage.objectAdmin", quiet=True)
print("Preparació feta.")

## 3. Exporta el dataset

Regenera `export/K`, `export/KP` i `export/KPS` des de `training-data/datasets/`. Cada conversa porta el missatge
de sistema fix de `training-data/missatge-sistema.txt`, i les fitxes reservades del benchmark en queden fora.

In [ ]:
print(sh([sys.executable, str(ROOT / "scripts" / "exporta_dataset.py")]) or "Export fet.")
print((ROOT / "export" / "INFORME.md").read_text(encoding="utf-8")[:1500])

## 4. Puja el run i engega la VM

In [ ]:
stage = ROOT / "runs" / RUN / "input"
if stage.exists():
    shutil.rmtree(stage)
(stage / "data").mkdir(parents=True)
for name in ("train.jsonl", "valid.jsonl"):
    shutil.copy(ROOT / "export" / VARIANT / name, stage / "data" / name)
shutil.copy(ROOT / "training" / "train.py", stage / "train.py")
(stage / "config.json").write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
gcloud("storage", "rsync", "-r", str(stage), f"{GS}/input")
print("Pujat a", GS)

In [ ]:
def create_vm(zone):
    args = [
        "compute", "instances", "create", VM, f"--zone={zone}",
        "--machine-type=g2-standard-8",          # 1x L4 (24 GB), 8 vCPU, 32 GB RAM
        "--maintenance-policy=TERMINATE",
        "--image-family=common-cu129-ubuntu-2204-nvidia-580",
        "--image-project=deeplearning-platform-release",
        "--boot-disk-size=200GB", "--boot-disk-type=pd-balanced",
        f"--service-account={SERVICE_ACCOUNT}", "--scopes=cloud-platform",
        f"--metadata=bucket={BUCKET},run={RUN},install-nvidia-driver=True",
        f"--metadata-from-file=startup-script={ROOT / 'training' / 'arrenca-vm.sh'}",
        f"--labels=maia=entrenament,run={RUN}",
        f"--max-run-duration={MAX_HOURS}h", "--instance-termination-action=STOP",
    ]
    if SPOT:
        args.append("--provisioning-model=SPOT")
    return gcloud(*args)

def create_anywhere():
    """Prova cada zona fins que una té una L4 lliure. Retorna la zona o None."""
    for zone in ZONES:
        try:
            create_vm(zone)
            return zone
        except RuntimeError as exc:
            reason = "sense capacitat" if "EXHAUSTED" in str(exc) or "STOCKOUT" in str(exc) else str(exc).splitlines()[-1]
            print(f"{zone}: {reason}")
    return None

ZONE = create_anywhere()
assert ZONE, "Cap zona amb capacitat ara mateix. Torna-ho a provar més tard o posa SPOT = False."
print(f"VM {VM} engegada a {ZONE}")

## 5. Segueix l'entrenament

Mostra l'estat de la VM i les últimes línies del log cada minut, fins que el run acaba (`DONE` o `FAILED`) o la VM
s'atura. El primer cop la VM tarda ~10 minuts a instal·lar Unsloth abans de començar a entrenar.

Nota: amb Gemma 4 E4B, Unsloth avisa que una pèrdua inicial de 13–15 és normal. El que compta és que baixi.

In [ ]:
def status():
    vm = gcloud("compute", "instances", "describe", VM, f"--zone={ZONE}", "--format=value(status)", check=False, quiet=True)
    marks = gcloud("storage", "ls", f"{GS}/", check=False, quiet=True)
    done = f"{GS}/DONE" in marks
    failed = f"{GS}/FAILED" in marks
    return vm or "ESBORRADA", done, failed

def tail(n=15):
    log = gcloud("storage", "cat", f"{GS}/train.log", check=False, quiet=True)
    return "\n".join(log.splitlines()[-n:])

while True:
    vm, done, failed = status()
    print(f"--- {datetime.datetime.now():%H:%M} · VM {vm} · {'DONE' if done else 'FAILED' if failed else 'en curs'}")
    print(tail())
    if done or failed or vm in ("TERMINATED", "STOPPED", "ESBORRADA"):
        break
    time.sleep(60)

### Reprèn (només si la VM spot s'ha aturat abans d'acabar)

Si l'estat és `TERMINATED` sense `DONE` ni `FAILED`, Google ha recuperat la VM spot. Aquesta cel·la la torna a
engegar; si la zona no té capacitat, l'esborra i en crea una de nova a la primera zona lliure de `ZONES`. Tot l'estat
és al bucket, així que l'script d'arrencada reprèn des de l'últim checkpoint pujat (cada 5 minuts).
Després torna a la cel·la anterior.

In [ ]:
vm, done, failed = status()
if vm == "TERMINATED" and not (done or failed):
    try:
        gcloud("compute", "instances", "start", VM, f"--zone={ZONE}")
        print("VM reengegada.")
    except RuntimeError as exc:
        print(f"No es pot reengegar a {ZONE} ({str(exc).splitlines()[-1]}). La recreo en una altra zona.")
        gcloud("compute", "instances", "delete", VM, f"--zone={ZONE}", "--quiet")
        ZONE = create_anywhere()
        assert ZONE, "Cap zona amb capacitat ara mateix. Torna-ho a provar més tard."
        print(f"VM {VM} recreada a {ZONE}")
else:
    print(f"No cal: VM {vm}, DONE={done}, FAILED={failed}")

## 6. Descarrega els resultats

In [ ]:
dest = ROOT / "runs" / RUN / "output"
dest.mkdir(parents=True, exist_ok=True)
gcloud("storage", "cp", f"{GS}/train.log", str(dest / "train.log"), check=False)
vm, done, failed = status()
if not done:
    # Sense DONE no hi ha adaptador: es baixa només el log per veure què ha passat.
    print(f"El run no ha acabat bé (VM {vm}, FAILED={failed}). Últimes línies del log:")
    print(tail(40))
    raise SystemExit("Revisa runs/<run>/output/train.log abans de tornar-ho a llançar.")
# Els checkpoints intermedis no cal baixar-los: l'adaptador final és a output/adapter.
gcloud("storage", "rsync", "-r", "--exclude=^checkpoints/", f"{GS}/output", str(dest))
metrics = json.loads((dest / "metrics.json").read_text(encoding="utf-8"))
print("Pèrdua final a valid:", metrics["final_eval"].get("eval_loss"))
evals = [(h["step"], round(h["eval_loss"], 4)) for h in metrics["log_history"] if "eval_loss" in h]
print("Corba de validació (pas, pèrdua):", evals)

In [ ]:
from IPython.display import Markdown, display
samples = dest / "mostres.md"
display(Markdown(samples.read_text(encoding="utf-8")) if samples.exists() else "No hi ha mostres.")

## 7. Esborra la VM

Una VM aturada ja no paga GPU, però el disc de 200 GB sí. Esborra-la quan tinguis els resultats.
Els fitxers del run es queden al bucket.

In [ ]:
gcloud("compute", "instances", "delete", VM, f"--zone={ZONE}", "--quiet")
print("VM esborrada.")

## Annex — VMs i runs existents

In [ ]:
print(gcloud("compute", "instances", "list", "--filter=labels.maia=entrenament",
             "--format=table(name,zone.basename(),status,scheduling.provisioningModel)", quiet=True) or "Cap VM.")
print(gcloud("storage", "ls", f"gs://{BUCKET}/runs/", check=False, quiet=True) or "Cap run.")